# Explore the catalog using interactive plotly graphs

In [1]:
import json
import numpy as np
import pandas as pd

import sys, os
from io import StringIO

import matplotlib.pyplot as plt
import plotly.express as px

from pathlib import Path

# Ensure project root is on sys.path so `import paths` finds the top-level paths.py
proj_root = Path('/Users/liekevanson/Documents/Projects/post_mt_review').resolve()
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))

from paths import DUMMY_CATALOG, MAIN_CATALOG

file_path = MAIN_CATALOG

In [35]:
# === Load JSON ===
def read_json_file(file_path): 
    try: 
        with open(file_path, "r") as f: data = json.load(f) 
        return data 
    except Exception as e: 
        print(f"Error reading file: {e}")
        return None

def read_json_file_topd(file_path, as_dataframe=True):
    try:
        with open(file_path, "r") as f:
            data = json.load(f)

        # If JSON is a dict of systems → convert to list
        if isinstance(data, dict):
            data = list(data.values())

        if as_dataframe:
            return pd.DataFrame(data)

        return data

    except Exception as e:
        print(f"Error reading file: {e}")
        return None


# ==  Function to easily extract some of the cols ==    
def extract_array(data, key):
    """Extracts a given key from each system and returns a NumPy array."""
    try:
        values = [entry.get(key, np.nan) for entry in data]
        return np.array(values)
    except Exception as e:
        print(f"Error extracting {key}: {e}")
        return None

def extract_multiple(data, keys):
    return {key: extract_array(data, key) for key in keys}

def to_triplet_array(arr):
    """
    Convert array-like of triplets to (N,3) float array.
    Invalid entries become [nan, nan, nan].
    """
    out = []
    for row in arr:
        if isinstance(row, (list, tuple, np.ndarray)) and len(row) == 3:
            out.append(row)
        else:
            out.append([np.nan, np.nan, np.nan])
    return np.array(out, dtype=float)
    

In [57]:
pd_catalog_data = read_json_file_topd(file_path)
print(pd_catalog_data.keys()    )
display(pd_catalog_data[pd_catalog_data["system_class"] == "Contact binary"])


Index(['System Name', 'Type1', 'Type2', 'Detection Method', 'Reference',
       'Notes', 'RA', 'Dec', 'Period', 'Eccentricity', 'M1', 'M1_sin3i', 'M2',
       'M2_sin3i', 'q', 'Mass Function', 'source_file', 'evol_type_1',
       'evol_type_2', 'obs_type_1', 'obs_type_2', 'system_class', 'Simbad'],
      dtype='object')


,System Name,Type1,Type2,Detection Method,Reference,Notes,RA,Dec,Period,Eccentricity,...,M2_sin3i,q,Mass Function,source_file,evol_type_1,evol_type_2,obs_type_1,obs_type_2,system_class,Simbad
148,LSS 3074,"O6-7:(f):, Contact","O4 f +, Contact","[RV, EB]",2017A&A...601A.133R,Gaia DR3 5868409430865830912,"[0.0097, 201.74924, 0.0097]","[0.0118, -62.03038, 0.0118]","[0.0006, 2.1852, 0.0006]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.04, 0.86, 0.04]","[None, None, None]",contact1.h5,MS,MS,O6-7:(f):,O4 f +,Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
149,MY Cam,"O4.5-6V, Contact","O6V, Contact","[RV, EB]",2014A&A...572A.110,Gaia DR3 469715181320008960,"[0.0236, 59.82622, 0.0236]","[0.0207, 57.23716, 0.0207]","[0.0, 1.17545, 0.0]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.04, 0.84, 0.04]","[None, None, None]",contact1.h5,MS,MS,O4.5-6V,O6V,Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
150,OGLE SMC-SC10 108086,"O9V, Contact","O9.5-B0V, Contact","[RV, EB]",2005MNRAS.357..304H,"Gaia DR3 4690510870334496896, SMC","[0.0242, 16.37751, 0.0242]","[0.0219, -72.0227, 0.0219]","[None, 0.8831, None]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.06, 0.85, 0.06]","[None, None, None]",contact1.h5,MS,MS,O9V,O9.5-B0V,Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
151,TU Mus,"O7V, Contact","O8V, Contact","[RV, EB, IUE]",2008ApJ...681..554P,"Gaia DR3 5237207155697930496, HD 100213","[0.0188, 172.79544, 0.0188]","[0.0195, -65.74225, 0.0195]","[0.0, 1.38728, 0.0]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.009, 0.623, 0.009]","[None, None, None]",contact1.h5,MS,MS,O7V,O8V,Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
152,V382 Cyg,"O6.5V((f)), Contact","O6V((f)), Contact","[RV, EB]",2017A&A...607A..82M,"Gaia DR3 2057530097777247744, HD 228854","[0.0183, 304.69673, 0.0183]","[0.024, 36.34055, 0.024]","[1e-05, 1.88555, 1e-05]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.005, 0.727, 0.005]","[None, None, None]",contact1.h5,MS,MS,O6.5V((f)),O6V((f)),Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
153,VFTS 352,"O4.5 V(n)((fc))z, Contact","O5.5 V(n)((fc))z, Contact","[RV, EB]",2015ApJ...812..102A,"Gaia DR3 4657678216202797440, LMC","[0.017, 84.61859, 0.017]","[0.0195, -69.18864, 0.0195]","[0.0, 1.12415, 0.0]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.1, 0.99, 0.1]","[None, None, None]",contact1.h5,MS,MS,O4.5 V(n)((fc))z,O5.5 V(n)((fc))z,Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
154,CT Tau,"B1V, Contact","B1V, Contact","[RV, EB]",2019AJ....157..111Y,"Gaia DR3 3430795519289951744,HD 249751","[0.021, 89.70881, 0.021]","[0.0182, 27.0783, 0.0182]","[0.0, 0.66683, 0.0]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.003, 0.983, 0.003]","[None, None, None]",contact1.h5,MS,MS,B1V,B1V,Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
155,GU Mon,"B1V, Contact","B1V, Contact","[RV, EB]",2019AJ....157..111Y,Gaia DR3 3125506151511957120,"[0.0172, 101.19525, 0.0172]","[0.0172, 0.22175, 0.0172]","[0.0, 0.89665, 0.0]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.003, 0.976, 0.003]","[None, None, None]",contact1.h5,MS,MS,B1V,B1V,Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
156,SV Cen,"B, Contact?","B, Contact?","[RV, EB]",1992AJ....103..573R,"Gaia DR3 5335388664983921024, HD 102552,Pdot s...","[0.0235, 176.98836, 0.0235]","[0.0222, -60.56604, 0.0222]","[0.001, 1.658, 0.001]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.002, 0.71, 0.002]","[None, None, None]",contact1.h5,MS,MS,B,B,Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
157,V606 Cen,"B0.5V, Contact","B2V, Contact","[RV, EB]","1999A&A...345..531L,2022ApJ...924...30L","Gaia DR3 5869546428991037312, HD 115937; tripl...","[0.0092, 200.40113, 0.0092]","[0.011, -60.52077, 0.011]","[0.0, 1.4951, 0.0]","[0.07, 0.33, 0.07]",...,"[None, None, None]","[0.0007, 0.5484, 0.0007]","[None, None, None]",contact1.h5,MS,MS,B0.5V,B2V,Contact binary,

In [58]:
pd_catalog_data.keys()

Index(['System Name', 'Type1', 'Type2', 'Detection Method', 'Reference',
       'Notes', 'RA', 'Dec', 'Period', 'Eccentricity', 'M1', 'M1_sin3i', 'M2',
       'M2_sin3i', 'q', 'Mass Function', 'source_file', 'evol_type_1',
       'evol_type_2', 'obs_type_1', 'obs_type_2', 'system_class', 'Simbad'],
      dtype='object')

In [ ]:
display(pd_catalog_data[['M1', 'M2', 'evol_type_1','evol_type_2', 'obs_type_1', 'obs_type_2',  'system_class',  ]])

M1 = to_triplet_array(pd_catalog_data['M1'] )
print(M1[:,1])  # Extract median M1 values
print(pd_catalog_data['M1'][:]) # Extract median M1 values

,M1,M2,evol_type_1,evol_type_2,obs_type_1,obs_type_2,system_class
0,"[None, None, None]","[None, None, None]",MS,WD,None,None,Blue straggler binary
1,"[None, None, None]","[None, None, None]",MS,WD,None,None,Blue straggler binary
2,"[None, None, None]","[None, None, None]",MS,WD,None,None,Blue straggler binary
3,"[None, None, None]","[None, None, None]",MS,WD,None,None,Blue straggler binary
4,"[None, None, None]","[None, None, None]",MS,WD,None,None,Blue straggler binary
...,...,...,...,...,...,...,...
3860,"[None, 1.75, None]","[None, 0.5, None]",MS,MS,A7V,[G6IV],Algol
3861,"[None, 2.0999999046325684, None]","[None, 0.5, None]",MS,MS,A5,[KIV],Algol
3862,"[None, 2.180000066757202, None]","[None, 1.059999942779541, None]",MS,MS,A3V,KIV,Algol
3863,"[0.5, 5.289999961853027, 0.5]","[0.25, 2.3299999237060547, 0.25]",MS,None,B3V-A2III,None,Algol


[       nan        nan        nan ... 2.18000007 5.28999996 6.59000015]
0                     [None, None, None]
1                     [None, None, None]
2                     [None, None, None]
3                     [None, None, None]
4                     [None, None, None]
                      ...               
3860                  [None, 1.75, None]
3861    [None, 2.0999999046325684, None]
3862     [None, 2.180000066757202, None]
3863       [0.5, 5.289999961853027, 0.5]
3864     [None, 6.590000152587891, None]
Name: M1, Length: 3865, dtype: object


In [70]:
# === Interactive Plotting with Plotly ===
def plotly_vars(col1, col2, catalog_data, data=None, logx=False, logy=False):
    """
    Plot col2 vs. col1 from catalog_data (e.g. output of extract_multiple),
    optionally using metadata (from JSON) for hover and coloring.
    a
    Parameters:
        col1, col2: str
            Keys to plot (e.g. 'M1', 'q')
        catalog_data: dict of str -> np.ndarray
            Typically from extract_multiple()
        data: list of dict
            Raw JSON data for metadata (optional but enables color, hover)
        logx, logy: bool
            Whether to use logarithmic axes
    """
    # --- sanitize inputs ---
    arr_x = to_triplet_array(catalog_data[col1])
    arr_y = to_triplet_array(catalog_data[col2])

    # mask rows with any NaNs
    mask = (
        np.isfinite(arr_x).all(axis=1) &
        np.isfinite(arr_y).all(axis=1)
    )

    arr_x = arr_x[mask]
    arr_y = arr_y[mask]


    # Extract central values and uncertainties
    x = arr_x[:, 1]
    xerr_lo = x - arr_x[:, 0]
    xerr_hi = arr_x[:, 2] - x

    y = arr_y[:, 1]
    yerr_lo = y - arr_y[:, 0]
    yerr_hi = arr_y[:, 2] - y

    # x = catalog_data[col1][:, 1]
    # xerr_lo = x - catalog_data[col1][:, 0]
    # xerr_hi = catalog_data[col1][:, 2] - x

    # y = catalog_data[col2][:, 1]
    # yerr_lo = y - catalog_data[col2][:, 0]
    # yerr_hi = catalog_data[col2][:, 2] - y

    if data is not None:
        data = [entry for entry, m in zip(data, mask) if m]

    # Default metadata
    N = len(x)
    system_name = [""] * N
    type1 = [""] * N
    type2 = ["Unknown"] * N
    marker_size = [4] * N

    if data is not None:
        system_name = [entry.get("System Name", "") for entry in data]
        type1 = [entry.get("Type1", "") for entry in data]
        type2 = [entry.get("Type2", "Unknown") for entry in data]
        evol_type1 = [entry.get("evol_type_1", "") for entry in data]
        evol_type2 = [entry.get("evol_type_2", "") for entry in data]
        sys_class = [entry.get("system_class", "") for entry in data]
        marker_size = [1 if t2 == "WD" else 5 for t2 in sys_class]

    fig = px.scatter(
        x=x,
        y=y,
        color=sys_class,
        hover_data={"System Name": system_name, "Type1": type1},
        error_x=xerr_hi,
        error_x_minus=xerr_lo,
        error_y=yerr_hi,
        error_y_minus=yerr_lo,
        size=marker_size,
        size_max=5,
        labels={"x": col1, "y": col2}
    )

    fig.update_layout(
        width=900,
        height=600,
        xaxis_title=col1,
        yaxis_title=col2,
        xaxis_title_font=dict(size=18),
        yaxis_title_font=dict(size=18),
        legend=dict(font=dict(size=16)),
    )

    fig.update_yaxes(tickfont=dict(size=14))
    fig.update_xaxes(tickfont=dict(size=14))

    if logx:
        fig.update_xaxes(type="log")
    if logy:
        fig.update_yaxes(type="log")

    return fig


In [71]:
# Load json data 
data = read_json_file(file_path) #"../data/post_mt_systems.json")
# show_available_keys
print(list(data[0].keys()))

# extract variables
catalog_data = extract_multiple(data, ["M1", "q", "Period", "Eccentricity" ])


['System Name', 'Type1', 'Type2', 'Detection Method', 'Reference', 'Notes', 'RA', 'Dec', 'Period', 'Eccentricity', 'M1', 'M1_sin3i', 'M2', 'M2_sin3i', 'q', 'Mass Function', 'source_file', 'evol_type_1', 'evol_type_2', 'obs_type_1', 'obs_type_2', 'system_class', 'Simbad']


In [72]:
# Plot q vs M1
plotly_vars("Period", "Eccentricity", catalog_data, data=data, logx=True)


In [75]:
P = to_triplet_array(pd_catalog_data['Period'] )
pd_catalog_data[P[:,1] < 1 ]

,System Name,Type1,Type2,Detection Method,Reference,Notes,RA,Dec,Period,Eccentricity,...,M2_sin3i,q,Mass Function,source_file,evol_type_1,evol_type_2,obs_type_1,obs_type_2,system_class,Simbad
150,OGLE SMC-SC10 108086,"O9V, Contact","O9.5-B0V, Contact","[RV, EB]",2005MNRAS.357..304H,"Gaia DR3 4690510870334496896, SMC","[0.0242, 16.37751, 0.0242]","[0.0219, -72.0227, 0.0219]","[None, 0.8831, None]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.06, 0.85, 0.06]","[None, None, None]",contact1.h5,MS,MS,O9V,O9.5-B0V,Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
154,CT Tau,"B1V, Contact","B1V, Contact","[RV, EB]",2019AJ....157..111Y,"Gaia DR3 3430795519289951744,HD 249751","[0.021, 89.70881, 0.021]","[0.0182, 27.0783, 0.0182]","[0.0, 0.66683, 0.0]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.003, 0.983, 0.003]","[None, None, None]",contact1.h5,MS,MS,B1V,B1V,Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
155,GU Mon,"B1V, Contact","B1V, Contact","[RV, EB]",2019AJ....157..111Y,Gaia DR3 3125506151511957120,"[0.0172, 101.19525, 0.0172]","[0.0172, 0.22175, 0.0172]","[0.0, 0.89665, 0.0]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.003, 0.976, 0.003]","[None, None, None]",contact1.h5,MS,MS,B1V,B1V,Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
158,V701 Sco,"B1-1.5V, Contact","B1-1.5V, Contact","[RV, EB]",2019AJ....157..111Y,"Gaia DR3 4054631788110501888, HD 317844","[0.0293, 263.60215, 0.0293]","[0.0219, -32.50445, 0.0219]","[0.0, 0.76187, 0.0]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.002, 0.995, 0.002]","[None, None, None]",contact1.h5,MS,MS,B1-1.5V,B1-1.5V,Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
161,RZ Pyx,"B5V, Contact","B5/7V, Contact","[RV, EB]",1987MNRAS.227..481B,"Gaia DR3 5648950448965966080, HD 75920","[0.0151, 133.0183, 0.0151]","[0.0209, -27.48372, 0.0209]","[3e-05, 0.65627, 3e-05]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.02, 0.82, 0.02]","[None, None, None]",contact1.h5,MS,MS,B5V,B5/7V,Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3829,RT Per,NaN,NaN,"[EB, SB2, RV]","[2020MNRAS.491.5489M, 2004yCat.5115....0S]",Semi-detached double-lined eclipsing binary co...,"[None, 50.91833, None]","[None, 46.57662, None]","[None, 0.84940032, None]","[None, None, None]",...,"[None, None, None]","[None, None, None]","[None, None, None]",NaN,MS,MS,F5V,G7IV,Algol,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
3846,U Sct,NaN,NaN,"[EB, SB2, RV]","[2020MNRAS.491.5489M, 2004yCat.5115....0S]",Semi-detached double-lined eclipsing binary co...,"[None, 283.61323, None]","[None, -12.60981, None]","[None, 0.95498575, None]","[None, None, None]",...,"[None, None, None]","[None, None, None]","[None, None, None]",NaN,MS,MS,F,[G7IV],Algol,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
3852,X Tri,NaN,NaN,"[EB, SB2, RV]","[2020MNRAS.491.5489M, 2004yCat.5115....0S]",Semi-detached double-lined eclipsing binary co...,"[None, 30.14057, None]","[None, 27.88867, None]","[None, 0.9715382, None]","[None, None, None]",...,"[None, None, None]","[None, None, None]","[None, None, None]",NaN,MS,MS,A3V,G5IV,Algol,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
3854,VV UMa,NaN,NaN,"[EB, SB2, RV]","[2020MNRAS.491.5489M, 2004yCat.5115....0S]",Semi-detached double-lined eclipsing binary co...,"[None, 144.528, None]","[None, 56.01869, None]","[None, 0.68735545, None]","[None, None, None]",...,"[None, None, None]","[None, None, None]","[None, None, None]",NaN,MS,MS,A1V,[G5IV],Algol,https://simbad.cds.unistra.fr/simbad/sim-coo?C...


In [ ]:
display(pd_catalog_data[pd_catalog_data['system_class'] == 'Contact binary' ])


,System Name,Type1,Type2,Detection Method,Reference,Notes,RA,Dec,Period,Eccentricity,...,M2_sin3i,q,Mass Function,source_file,evol_type_1,evol_type_2,obs_type_1,obs_type_2,system_class,Simbad
148,LSS 3074,"O6-7:(f):, Contact","O4 f +, Contact","[RV, EB]",2017A&A...601A.133R,Gaia DR3 5868409430865830912,"[0.0097, 201.74924, 0.0097]","[0.0118, -62.03038, 0.0118]","[0.0006, 2.1852, 0.0006]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.04, 0.86, 0.04]","[None, None, None]",contact1.h5,MS,MS,O6-7:(f):,O4 f +,Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
149,MY Cam,"O4.5-6V, Contact","O6V, Contact","[RV, EB]",2014A&A...572A.110,Gaia DR3 469715181320008960,"[0.0236, 59.82622, 0.0236]","[0.0207, 57.23716, 0.0207]","[0.0, 1.17545, 0.0]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.04, 0.84, 0.04]","[None, None, None]",contact1.h5,MS,MS,O4.5-6V,O6V,Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
150,OGLE SMC-SC10 108086,"O9V, Contact","O9.5-B0V, Contact","[RV, EB]",2005MNRAS.357..304H,"Gaia DR3 4690510870334496896, SMC","[0.0242, 16.37751, 0.0242]","[0.0219, -72.0227, 0.0219]","[None, 0.8831, None]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.06, 0.85, 0.06]","[None, None, None]",contact1.h5,MS,MS,O9V,O9.5-B0V,Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
151,TU Mus,"O7V, Contact","O8V, Contact","[RV, EB, IUE]",2008ApJ...681..554P,"Gaia DR3 5237207155697930496, HD 100213","[0.0188, 172.79544, 0.0188]","[0.0195, -65.74225, 0.0195]","[0.0, 1.38728, 0.0]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.009, 0.623, 0.009]","[None, None, None]",contact1.h5,MS,MS,O7V,O8V,Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
152,V382 Cyg,"O6.5V((f)), Contact","O6V((f)), Contact","[RV, EB]",2017A&A...607A..82M,"Gaia DR3 2057530097777247744, HD 228854","[0.0183, 304.69673, 0.0183]","[0.024, 36.34055, 0.024]","[1e-05, 1.88555, 1e-05]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.005, 0.727, 0.005]","[None, None, None]",contact1.h5,MS,MS,O6.5V((f)),O6V((f)),Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
153,VFTS 352,"O4.5 V(n)((fc))z, Contact","O5.5 V(n)((fc))z, Contact","[RV, EB]",2015ApJ...812..102A,"Gaia DR3 4657678216202797440, LMC","[0.017, 84.61859, 0.017]","[0.0195, -69.18864, 0.0195]","[0.0, 1.12415, 0.0]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.1, 0.99, 0.1]","[None, None, None]",contact1.h5,MS,MS,O4.5 V(n)((fc))z,O5.5 V(n)((fc))z,Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
154,CT Tau,"B1V, Contact","B1V, Contact","[RV, EB]",2019AJ....157..111Y,"Gaia DR3 3430795519289951744,HD 249751","[0.021, 89.70881, 0.021]","[0.0182, 27.0783, 0.0182]","[0.0, 0.66683, 0.0]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.003, 0.983, 0.003]","[None, None, None]",contact1.h5,MS,MS,B1V,B1V,Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
155,GU Mon,"B1V, Contact","B1V, Contact","[RV, EB]",2019AJ....157..111Y,Gaia DR3 3125506151511957120,"[0.0172, 101.19525, 0.0172]","[0.0172, 0.22175, 0.0172]","[0.0, 0.89665, 0.0]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.003, 0.976, 0.003]","[None, None, None]",contact1.h5,MS,MS,B1V,B1V,Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
156,SV Cen,"B, Contact?","B, Contact?","[RV, EB]",1992AJ....103..573R,"Gaia DR3 5335388664983921024, HD 102552,Pdot s...","[0.0235, 176.98836, 0.0235]","[0.0222, -60.56604, 0.0222]","[0.001, 1.658, 0.001]","[0.0, 0.0, 0.0]",...,"[None, None, None]","[0.002, 0.71, 0.002]","[None, None, None]",contact1.h5,MS,MS,B,B,Contact binary,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
157,V606 Cen,"B0.5V, Contact","B2V, Contact","[RV, EB]","1999A&A...345..531L,2022ApJ...924...30L","Gaia DR3 5869546428991037312, HD 115937; tripl...","[0.0092, 200.40113, 0.0092]","[0.011, -60.52077, 0.011]","[0.0, 1.4951, 0.0]","[0.07, 0.33, 0.07]",...,"[None, None, None]","[0.0007, 0.5484, 0.0007]","[None, None, None]",contact1.h5,MS,MS,B0.5V,B2V,Contact binary,

In [77]:
# Unique references for contact binaries
print(np.unique(pd_catalog_data['Reference'][pd_catalog_data['system_class'] == 'Contact binary' ]))

['1984AJ.....89..872L' '1985MNRAS.213...75H' '1987MNRAS.227..481B'
 '1992AJ....103..573R' '1999A&A...345..531L,2022ApJ...924...30L'
 '2005MNRAS.357..304H' '2008ApJ...681..554P' '2013A&A...559A..22M'
 '2014A&A...572A.110' '2014MNRAS.442.1560C' '2014NewA...31...32Y'
 '2015ApJ...812..102A' '2017A&A...601A.133R' '2017A&A...606A..54L'
 '2017A&A...607A..82M' '2019AJ....157..111Y' '2020A&A...634A.119M'
 '2022ApJ...932...14L']
